# Validation Set Visualization

This notebook loads a saved checkpoint, reconstructs the exact validation split from the saved `split_summary`, runs inference on the validation set, and visualizes predictions against ground-truth masks.

Use `checkpoint_best.pt` for the cleanest validation view. `model_refit_full.pt` is useful too, but it has already been retrained on the full dataset.


In [ ]:
from pathlib import Path
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from tqdm.auto import tqdm

from train_car_part_damage import (
    CLASS_NAMES,
    IMAGENET_MEAN,
    IMAGENET_STD,
    CarPartSegmentationDataset,
    build_model,
    build_splits_from_summary,
)

CHECKPOINT_PATH = Path('artifacts/car_part_damage/checkpoint_best.pt')
DATASET_NAME = 'moondream/car_part_damage'
NUM_EXAMPLES = 8
VIEW_MODE = 'worst'  # one of: worst, best, random, sequential
SEED = 42
ALPHA = 0.45
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('checkpoint:', CHECKPOINT_PATH)
print('device:', DEVICE)


In [ ]:
def denormalize_image(image_t: torch.Tensor) -> np.ndarray:
    image = image_t.detach().cpu().permute(1, 2, 0).numpy()
    mean = np.array(IMAGENET_MEAN, dtype=np.float32)
    std = np.array(IMAGENET_STD, dtype=np.float32)
    image = (image * std) + mean
    return np.clip(image, 0.0, 1.0)


def make_palette(num_classes: int) -> np.ndarray:
    tab20 = (plt.cm.tab20(np.linspace(0, 1, 20))[:, :3] * 255).astype(np.uint8)
    tab20b = (plt.cm.tab20b(np.linspace(0, 1, 20))[:, :3] * 255).astype(np.uint8)
    colors = [np.array([0, 0, 0], dtype=np.uint8)]
    colors.extend(tab20)
    colors.extend(tab20b)
    return np.stack(colors[:num_classes], axis=0)


PALETTE = make_palette(len(CLASS_NAMES))


def colorize_mask(mask: np.ndarray) -> np.ndarray:
    return PALETTE[mask]


def overlay_mask(image: np.ndarray, color_mask: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    image_u8 = (image * 255).astype(np.uint8)
    overlay = image_u8.copy()
    foreground = color_mask.sum(axis=-1) > 0
    overlay[foreground] = (
        (1.0 - alpha) * image_u8[foreground] + alpha * color_mask[foreground]
    ).astype(np.uint8)
    return overlay


def compute_foreground_iou(pred_mask: np.ndarray, target_mask: np.ndarray) -> float:
    class_ids = sorted(set(np.unique(pred_mask)).union(set(np.unique(target_mask))))
    class_ids = [class_id for class_id in class_ids if class_id != 0]
    if not class_ids:
        return float('nan')

    scores = []
    for class_id in class_ids:
        pred = pred_mask == class_id
        target = target_mask == class_id
        union = np.logical_or(pred, target).sum()
        if union == 0:
            continue
        inter = np.logical_and(pred, target).sum()
        scores.append(inter / union)
    return float(np.mean(scores)) if scores else float('nan')


def select_examples(records, mode: str, num_examples: int, seed: int):
    if mode == 'sequential':
        return records[:num_examples]
    if mode == 'best':
        return sorted(records, key=lambda item: item['foreground_iou'], reverse=True)[:num_examples]
    if mode == 'worst':
        return sorted(records, key=lambda item: item['foreground_iou'])[:num_examples]
    if mode == 'random':
        rng = random.Random(seed)
        return rng.sample(records, k=min(num_examples, len(records)))
    raise ValueError(f'Unknown VIEW_MODE: {mode}')


def present_classes(mask: np.ndarray):
    class_ids = [class_id for class_id in np.unique(mask) if class_id != 0]
    return [CLASS_NAMES[class_id] for class_id in class_ids]


In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')
split_summary = checkpoint['split_summary']
model_config = checkpoint['model_config']

dataset = load_dataset(DATASET_NAME)
_, val_split = build_splits_from_summary(dataset, split_summary)
val_dataset = CarPartSegmentationDataset(
    val_split,
    image_size=model_config['image_size'],
    train=False,
    hflip_prob=0.0,
)

model = build_model(model_config['encoder_name'], encoder_weights=None)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(DEVICE)
model.eval()

print('validation images:', len(val_dataset))
print('checkpoint epoch:', checkpoint.get('epoch', 'n/a'))
print('split strategy:', split_summary['strategy'])


In [ ]:
records = []

with torch.inference_mode():
    for idx in tqdm(range(len(val_dataset)), desc='running validation inference'):
        image_t, target_mask_t = val_dataset[idx]
        logits = model(image_t.unsqueeze(0).to(DEVICE))
        pred_mask = logits.argmax(dim=1)[0].cpu().numpy().astype(np.uint8)
        target_mask = target_mask_t.cpu().numpy().astype(np.uint8)
        image_np = denormalize_image(image_t)
        fg_iou = compute_foreground_iou(pred_mask, target_mask)
        records.append({
            'index': idx,
            'image': image_np,
            'target_mask': target_mask,
            'pred_mask': pred_mask,
            'foreground_iou': fg_iou,
            'classes': present_classes(target_mask),
            'image_id': val_split[idx].get('image_id', f'val_{idx:04d}'),
        })

valid_scores = [r['foreground_iou'] for r in records if not math.isnan(r['foreground_iou'])]
print('mean foreground IoU over images:', np.mean(valid_scores) if valid_scores else float('nan'))
selected = select_examples(records, VIEW_MODE, NUM_EXAMPLES, SEED)
print('displaying', len(selected), 'examples in', VIEW_MODE, 'mode')


In [ ]:
if not selected:
    raise RuntimeError('No validation examples were selected for display')

fig, axes = plt.subplots(len(selected), 4, figsize=(20, 5 * len(selected)))
if len(selected) == 1:
    axes = np.expand_dims(axes, axis=0)

for row, record in enumerate(selected):
    image = record['image']
    gt_rgb = colorize_mask(record['target_mask'])
    pred_rgb = colorize_mask(record['pred_mask'])
    gt_overlay = overlay_mask(image, gt_rgb, alpha=ALPHA)
    pred_overlay = overlay_mask(image, pred_rgb, alpha=ALPHA)
    error_map = record['pred_mask'] != record['target_mask']

    axes[row, 0].imshow(image)
    axes[row, 0].set_title(f"image\n{record['image_id']}")
    axes[row, 1].imshow(gt_overlay)
    axes[row, 1].set_title('ground truth overlay')
    axes[row, 2].imshow(pred_overlay)
    axes[row, 2].set_title(f"prediction overlay\nfg IoU={record['foreground_iou']:.3f}")
    axes[row, 3].imshow(image)
    axes[row, 3].imshow(error_map, cmap='Reds', alpha=0.55)
    axes[row, 3].set_title('error map')

    class_text = ', '.join(record['classes']) if record['classes'] else 'background only'
    axes[row, 0].set_xlabel(class_text)

    for col in range(4):
        axes[row, col].axis('off')

plt.tight_layout()
plt.show()
